# Chapter 2 · Counting beats guessing

The simplest model that uses context: look at the previous character, and predict the
next one by counting what came after it in the training text. No neural network and no
training loop, just counting. Its score, **2.48**, is the number every later model has
to beat.

| | |
|---|---|
| **Open box** | the counting bigram, `tinygpt/bigram.py` |
| **The number** | 3.35 → 2.48 |
| **Time** | about an hour |
| **Python you need** | `for` loops, NumPy arrays and indexing |
| **GPU** | not needed for this chapter |

**How this chapter works.** Five steps, in order:

1. **Predict** `everyone`: guess before you run anything. A wrong guess is useful: it shows you what you expected.
2. **Run** `everyone`: run the cells and compare with your guesses.
3. **Investigate** `everyone`: open the box. Read the code that just ran and answer a few questions.
4. **Modify** `most people`: change one thing, predict what will happen, then explain what did.
5. **Make** `optional challenge`: build this chapter's piece yourself. A check tells you when it's right.

Run a cell with **Shift+Enter**. If the notebook restarts, run the setup cell again first.

## Setup

In [ ]:
#@title Setup: run this cell first (about 30 seconds) { display-mode: "form" }
# Downloads tinyGPT (on Colab) and installs it. You don't need to read or change this cell.
import importlib, os, subprocess, sys
from pathlib import Path

def run(*command):
    done = subprocess.run(command, capture_output=True, text=True)
    if done.returncode != 0:
        raise RuntimeError(f"{' '.join(command)} failed:\n{done.stdout}{done.stderr}")

if "google.colab" in sys.modules:
    root = Path("/content/tinyGPT")
    if not root.exists():
        run("git", "clone", "-q", "https://github.com/cgpan/tinyGPT.git", str(root))
else:
    here = Path.cwd().resolve()
    root = next((p for p in [here, *here.parents] if (p / "src" / "tinygpt").is_dir()), None)
    if root is None:
        raise FileNotFoundError("Can't find the tinyGPT folder. Open this notebook from inside it, or use Colab.")
os.chdir(root)
run(sys.executable, "-m", "pip", "install", "-q", ".")
importlib.invalidate_caches()

import tinygpt
tinygpt.print_setup()

## 1 · Predict `everyone`

The model will count, for every character, which characters came right after it. Guess:

In [ ]:
guess_after_q = None        # which character most often comes right after "q"?
guess_after_period = None   # ... right after "."?
guess_loss = None           # its validation loss: somewhere between 0 and 4.17.
                            # (Chapter 1's frequency model got 3.35.)

And one more, in words: will text written by this model look like English? Like Shakespeare?

*Your explanation (double-click to edit):*

## 2 · Run `everyone`

`load_data` does chapter 1's work for you: tokenize the text and split it, 90% for
training and 10% for validation. The numbers come back as *tensors*, PyTorch's version
of a NumPy array.

In [ ]:
tok, train_ids, val_ids = tinygpt.load_data()
print(f"{len(train_ids):,} training characters, {len(val_ids):,} validation characters")
print(train_ids[:20])
print(repr(tok.decode(train_ids[:20])))

Now count every neighbouring pair in the training text. It takes about a second.

In [ ]:
model = tinygpt.CountBigram(tok.vocab_size).fit(train_ids)

Every square below is one pair. Rows are "this character", columns are "the next
character", and darker means more often. Before you look closely: where do you expect the
dark squares to be?

In [ ]:
tinygpt.bigram_heatmap(model.counts, tok)

Check your first two guesses:

In [ ]:
def likeliest_next(ch, k=5):
    probs = model.probs()[tok.stoi[ch]]
    best = probs.argsort()[::-1][:k]
    return [(tok.chars[i], round(float(probs[i]), 3)) for i in best]

print('after "q":', likeliest_next("q"))
print('after ".":', likeliest_next("."))
print("your guesses:", guess_after_q, guess_after_period)

Let the model write. Each new character is drawn using only the one before it.

In [ ]:
print(tok.decode(model.generate(300, seed=0)))

And the number, on the validation text the counts never saw:

In [ ]:
train_loss = model.loss(train_ids)
val_loss = model.loss(val_ids)
print(f"training {train_loss:.4f}   validation {val_loss:.4f}   your guess {guess_loss}")

## 3 · Investigate `everyone`

### Open the box

In [ ]:
tinygpt.show(tinygpt.CountBigram)

1. **`fit`**: in plain words, what does the loop do? How many pairs does it count in a
   text of *n* characters? Check with the next cell.
2. **`probs`**: every "q" in the training text is followed by "u". So why does the model
   give "u" after "q" only about 90%? Look for `alpha`.
3. **`loss`**: it averages −log p over every neighbouring pair. Work out the surprise of
   one pair, "u" after "q", in the cell below.
4. **`generate`**: each character depends only on the one before. So why do real words
   like "the" and "my" appear?
5. Why is the training loss lower than the validation loss?

In [ ]:
print(f"pairs counted: {model.counts.sum():,.0f}   characters in the training text: {len(train_ids):,}")

q, u = tok.stoi["q"], tok.stoi["u"]
print(f'"q" appears {model.counts[q].sum():.0f} times, followed by "u" {model.counts[q, u]:.0f} times')
print(f'the model\'s probability of "u" after "q": {model.probs()[q, u]:.3f}')

import math
print(f'surprise at "u" after "q": {-math.log(model.probs()[q, u]):.3f} nats')

*Your explanation (double-click to edit):*

**Perplexity** is another way to say the same number: e to the power of the loss. It
reads as "on average, as unsure as picking among this many characters".

In [ ]:
print(f"knowing nothing: {math.exp(4.1744):.1f}   counting pairs: {math.exp(val_loss):.1f}")

## 4 · Modify `most people`

**A. Smoothing.** `alpha` pretends every pair was seen a few extra times. **Predict** the
validation loss for `alpha = 0` (no pretending) and for `alpha = 100` (a lot), then run.

In [ ]:
for alpha in [1.0, 0.01, 0.0, 100.0]:
    m = tinygpt.CountBigram(tok.vocab_size, alpha=alpha)
    m.counts = model.counts          # the same counts; only the smoothing changes
    print(f"alpha = {alpha:>6}: validation loss {m.loss(val_ids):.4f}")

`inf` means infinitely surprised. Which pairs did that? This cell lists pairs in the
validation text that never appear in the training text:

In [ ]:
pairs = zip(val_ids[:-1].tolist(), val_ids[1:].tolist())
unseen = [tok.chars[a] + tok.chars[b] for a, b in pairs if model.counts[a, b] == 0]
print(len(unseen), "unseen pairs, for example:", unseen[:12])

Explain the `inf` in one or two sentences. What is the difference between "never seen"
and "impossible"? And why does `alpha = 100` do worse than `alpha = 1`?

*Your explanation (double-click to edit):*

**B. Less data.** **Predict:** how much worse is the model if it counts only the first
1,000 characters? 10,000? 100,000?

In [ ]:
for n in [1_000, 10_000, 100_000, len(train_ids)]:
    smaller = tinygpt.CountBigram(tok.vocab_size).fit(train_ids[:n])
    print(f"counted {n:>9,} characters: validation loss {smaller.loss(val_ids):.4f}")

*Your explanation (double-click to edit):*

## 5 · Make `optional challenge`

**Part 1.** Write the counting yourself. Fill in `count_pairs` and run the cell; the
check compares your table with tinyGPT's.

In [ ]:
import numpy as np

def count_pairs(ids, vocab_size):
    """Return a table where counts[a, b] = how many times b comes right after a in ids."""
    counts = np.zeros((vocab_size, vocab_size))
    # your code here: look at every neighbouring pair (a, b) in ids and add 1 to counts[a, b]
    return counts

tinygpt.check.count_pairs(count_pairs)

Now put your table inside tinyGPT's model in place of its own counts. If your counting is
right, you get exactly the same validation loss as above. (An empty table gives 4.17:
counting nothing is knowing nothing.)

In [ ]:
mine = tinygpt.CountBigram(tok.vocab_size)
mine.counts = count_pairs(train_ids.tolist(), tok.vocab_size)
print(f"your counts: validation loss {mine.loss(val_ids):.4f}")

**Part 2: two characters of memory.** Count *triples*: how often character c follows the
pair a, b. The table is 65 × 65 × 65. `triple_loss` is `CountBigram.loss` stretched to
three characters; compare the two.

In [ ]:
def count_triples(ids, vocab_size):
    """Return a table where counts[a, b, c] = how many times c comes right after a, b."""
    counts = np.zeros((vocab_size, vocab_size, vocab_size))
    # your code here
    return counts

tinygpt.check.count_triples(count_triples)


def triple_loss(counts, ids, alpha=0.1):
    """Average surprise when each prediction uses the two characters before it."""
    smoothed = counts + alpha
    probs = smoothed / smoothed.sum(axis=2, keepdims=True)
    ids = np.asarray(ids)
    p = probs[ids[:-2], ids[1:-1], ids[2:]]
    return float(-np.log(p).mean())

triples = count_triples(train_ids.tolist(), tok.vocab_size)
print(f"two characters of memory: validation loss {triple_loss(triples, val_ids):.4f}")

**Stuck?** Try first, then compare with [the solution](https://github.com/cgpan/tinyGPT/blob/main/solutions/02_counting.md) (also in the `solutions/` folder of your copy).

### Why not just count longer contexts? `everyone`

More memory lowers the number: one character of memory gives 2.48, two give 2.05. So why
not count with 10 characters of memory? Look at the size of the table, where `memory` is
how many characters the model looks back at:

In [ ]:
for memory in [1, 2, 3, 4, 8, 128]:
    print(f"memory {memory:>3}: a table with {65 ** (memory + 1):.2e} cells")

Four characters of memory already needs about 9 GB. tinyGPT looks back 128 characters;
that table would have a number of cells 234 digits long. And nearly every cell would be
empty: one million characters of training text filled only 1,380 of the 4,225 pair cells,
and 11,228 of the 274,625 triple cells.

Counting has hit a wall. From chapter 3 on, the model **learns** instead of counting.

## Where the number stands

In [ ]:
results = {"knowing frequencies (chapter 1)": 3.35, "counting pairs": val_loss}
# Did Part 2 of the challenge? Add it:
# results["counting triples"] = triple_loss(triples, val_ids)
tinygpt.scoreboard(results)

## Common questions

**Is this how ChatGPT works?** It does the same job: give a probability to every possible
next piece of text. But it looks back over thousands of tokens instead of one character,
and it learns its probabilities instead of counting them.

**Why "bigram"?** *Bi* means two: the model is built from pairs. Triples make a trigram.

**Why `alpha = 1`?** Without smoothing, one unseen pair makes the loss infinite. With far
too much, the real counts get drowned out. Here 1 happens to work well, and 0.1 is almost
the same.

**Why is the training loss lower than the validation loss?** The counts came from the
training text, so the model has, in a sense, seen those answers before.

## If something breaks

| What you see | What to do |
|---|---|
| `RuntimeWarning: divide by zero encountered in log` | Expected with `alpha = 0`: a probability of zero has infinite surprise. |
| Your `count_pairs` takes more than a few seconds | Pass a list, not a tensor: `train_ids.tolist()`, as in the cell. |
| "Your session crashed after using all available RAM" | A table with too many cells: five characters already needs 9 GB. Restart, run the setup cell, and stay at three or fewer. |
| `NameError` | Run the cells above it, in order. In Colab: Runtime > Run before. |

## Next

**Chapter 3 · Learning instead of counting.** A model that learns the same table by
gradient descent, and why learning can grow where counting cannot. You won't need a GPU
until chapter 5.